In [ ]:
# Requirements

# using Pkg
# Pkg.add("Lux")
# Pkg.add("Random")
# Pkg.add("Zygote")
# Pkg.add("Optimisers")
# Pkg.add("Plots")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [50]:
using Lux

input_dim = 8
output_dim = 4
model = Dense(input_dim, output_dim)

Dense(8 => 4)       # 36 parameters

In [51]:
using Lux
input_dim = 2
intermediate_dim = 16
output_dim = 1

# 2 => 16 => relu => 1
model = Chain(
    Dense(input_dim, intermediate_dim, relu),
    Dense(intermediate_dim, output_dim),
)

Chain(
    layer_1 = Dense(2 => 16, relu),     # 48 parameters
    layer_2 = Dense(16 => 1),           # 17 parameters
)         # Total: 65 parameters,
          #        plus 0 states.

In [52]:
using Lux, Random

input_dim = 8
output_dim = 4
# `|> f32` casts all the numeric values to Float32
model = Dense(input_dim, output_dim) |> f32

rng = Random.default_rng() # you can choose RNG as you like
rng = Random.seed!(rng, 40) # fix the rng seed
ps, st = Lux.setup(rng, model)

vector_input = [1, 2, 3, 4, 5, 6, 7, 8] |> f32
output, new_st = model(vector_input, ps, st)

(Float32[5.2590766, 3.3526256, 5.555236, -1.923615], NamedTuple())

In [53]:
using Lux, Random, Zygote

# Zygote may not work properly in global scope
function main()
    input_dim = 8
    output_dim = 8
    # `|> f32` casts all the numeric values to Float32
    model = Dense(input_dim, output_dim) |> f32

    rng = Random.default_rng() # you can choose RNG as you like
    rng = Random.seed!(rng, 40) # fix the rng seed
    ps, st = Lux.setup(rng, model)

    function mse_loss(y_true, y_hat)
        return sum((y_true - y_hat).^2)
    end

    vector_input = [1, 2, 3, 4, 5, 6, 7, 8] |> f32
    y_true = vector_input # try to learn the identity function (y = x)
    grads = Zygote.gradient(
        ps -> begin
            # THIS MUST BE INCLUDED IN THE SCOPE!!!
            y_hat, new_st = model(vector_input, ps, st)
            st = new_st
            mse_loss(y_true, y_hat)
        end,
        ps
    )[1]
end

main()


(weight = Float32[10.083963 20.167927 … 70.587746 80.67171; -10.51483 -21.02966 … -73.603806 -84.11864; … ; -2.1953497 -4.3906994 … -15.367448 -17.562798; -8.623651 -17.247301 … -60.365555 -68.989204], bias = Float32[10.083963, -10.51483, -15.130152, 3.7492723, -0.041480064, -10.63965, -2.1953497, -8.623651])

In [54]:
using Optimisers

learning_rate = 1e-4
opt = Optimisers.OptimiserChain(
    ClipNorm(1.0),
    Adam(learning_rate),
)

OptimiserChain(ClipNorm{Float64, Float64}(1.0, 2.0, true), Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8))

In [55]:
using Lux, Random, Zygote, Optimisers

# Zygote may not work properly in global scope
function main()
    input_dim = 8
    output_dim = 8
    # `|> f32` casts all the numeric values to Float32
    model = Dense(input_dim, output_dim) |> f32

    rng = Random.default_rng() # you can choose RNG as you like
    rng = Random.seed!(rng, 40) # fix the rng seed
    ps, st = Lux.setup(rng, model)

    learning_rate = 1e-4
    opt = Optimisers.OptimiserChain(
        ClipNorm(1.0),
        Adam(learning_rate),
    )
    opt_state = Optimisers.setup(opt, ps)

    function mse_loss(y_true, y_hat)
        return sum((y_true - y_hat).^2)
    end

    vector_input = [1, 2, 3, 4, 5, 6, 7, 8] |> f32
    y_true = vector_input # try to learn the identity function (y = x)
    grads = Zygote.gradient(
        ps -> begin
            # THIS MUST BE INCLUDED IN THE SCOPE!!!
            y_hat, new_st = model(vector_input, ps, st)
            st = new_st
            mse_loss(y_true, y_hat)
        end,
        ps
    )[1]

    display(ps) # Display parameters before update
    opt_state, ps = Optimisers.update(opt_state, ps, grads)
    display(ps) # Display parameters after update
end

main()


(weight = Float32[-0.33398467 0.43204907 … 0.55984545 0.5002263; -0.029455915 -0.099705815 … 0.3199415 -0.15672576; … ; -0.48615417 0.07889257 … 0.07015245 0.11924155; 0.4749607 -0.31358016 … -0.33515924 0.38729846], bias = Float32[0.22096232, -0.29122669, 0.12903713, -0.3231772, 0.119143665, 0.13592169, -0.23593734, 0.3118714])

(weight = Float32[-0.33408466 0.43194908 … 0.55974543 0.5001263; -0.029355915 -0.09960581 … 0.32004148 -0.15662576; … ; -0.48605418 0.078992575 … 0.07025245 0.11934155; 0.4750607 -0.31348017 … -0.33505926 0.38739845], bias = Float32[0.22086231, -0.2911267, 0.12913713, -0.32327718, 0.11924367, 0.13602169, -0.23583734, 0.3119714])

In [ ]:
using Lux, Random, Zygote, Optimisers, Plots

function mse_loss(y_true, y_hat)
    return sum((y_true - y_hat).^2)
end

function main()
    input_dim = 1
    intermediate_dim = 8
    output_dim = 1
    # `|> f32` casts all the numeric values to Float32
    model = Chain(
        Dense(input_dim, intermediate_dim, relu) |> f32,
        Dense(intermediate_dim, output_dim) |> f32,
    )

    rng = Random.default_rng() # you can choose RNG as you like
    rng = Random.seed!(rng, 40) # fix the rng seed
    ps, st = Lux.setup(rng, model)

    learning_rate = 1e-3
    opt = Optimisers.OptimiserChain(
        ClipNorm(1.0),
        Adam(learning_rate),
    )
    opt_state = Optimisers.setup(opt, ps)

    anim = Animation()

    xs = collect(0:0.01:2*pi) |> f32
    num_sample = length(xs)
    batch_size = min(16, num_sample ÷ 8)
    episode = 0
    while true
        # validation
        y_hat, _ = model(reshape(xs, 1, :, 1), ps, st)
        mse_loss(xs, y_hat[:]) < 0.5 && break

        # plot
        plot(xs, xs, label="ideal")
        plt = plot!(xs, y_hat[:], label="model", title="episode $episode")
        frame(anim, plt)

        # training process
        episode = episode + 1
        x = reshape(rand(rng, xs, batch_size), :, 1, batch_size)
        y_hat, st = model(x, ps, st)
        grads, = Zygote.gradient(ps -> begin
                y_hat, _ = model(x, ps, st)
                mse_loss(x, y_hat)
            end,
            ps
        )
        opt_state, ps = Optimisers.update(opt_state, ps, grads)

        # quit condition
        episode > 5000 && break
    end
    # plot
    y_hat, _ = model(reshape(xs, 1, :, 1), ps, st)
    plot(xs, xs[:], label="ideal")
    plt = plot!(xs, y_hat[:], label="model", title="episode $episode")
    frame(anim, plt)

    return anim
end

anim = main()


Animation("/tmp/jl_XdNk0R", ["000001.png", "000002.png", "000003.png", "000004.png", "000005.png", "000006.png", "000007.png", "000008.png", "000009.png", "000010.png"  …  "001333.png", "001334.png", "001335.png", "001336.png", "001337.png", "001338.png", "001339.png", "001340.png", "001341.png", "001342.png"])

In [86]:
gif(anim, fps=8)
mp4(anim, fps=8)

[ Info: Saved animation to /mnt/c/Users/tommy/Desktop/tomlog/source/_drafts/lux-log/tmp.gif
[ Info: Saved animation to /mnt/c/Users/tommy/Desktop/tomlog/source/_drafts/lux-log/tmp.mp4


Plots.AnimatedGif("/mnt/c/Users/tommy/Desktop/tomlog/source/_drafts/lux-log/tmp.mp4")